# Lab 5: Product Review Sentiment Classification

**Task:** classify Amazon product reviews as positive or negative from their text.

**Dataset:** Amazon Review Polarity (https://s3.amazonaws.com/fast-ai-nlp/amazon_review_polarity_csv.tgz).

> **The data is not in this repository: the download is about 690 MB (1.7 GB unpacked), far too big for git. Download it before the lab** (it takes a while), unpack it into this lab's `data/` folder, so that the files are `data/amazon_review_polarity_csv/train.csv` and `test.csv`.

3.6 million training and 400,000 test reviews. Each row has a label (1 = negative, 1–2 stars; 2 = positive, 4–5 stars; 3-star reviews were left out), the review title and the review text. We work with a random sample so that the lab runs in minutes.

**What we will do:**
1. Load reviews and labels.
2. Build a text preprocessing pipeline: lowercase, remove punctuation, remove stopwords, stem.
3. Represent each review with TF-IDF and understand why it works.
4. Train Naive Bayes, logistic regression and a linear SVM.
5. Evaluate with precision, recall and F1, and study misclassified reviews.
6. Report the most positive and negative words, and the effect of each preprocessing step (an ablation study).

In [ ]:
import re
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.width", 150)
pd.set_option("display.max_colwidth", 120)

## 1. Loading a sample

The files have no header row. Reading all 3.6 million rows would take a lot of memory; we read the first 600,000 training rows and take a random sample of 60,000 for training, and 20,000 from the test file. (Check that the first rows are not sorted by label: they aren't, but in general don't assume it.)

In [ ]:
cols = ["label", "title", "text"]
train_full = pd.read_csv("data/amazon_review_polarity_csv/train.csv", header=None, names=cols, nrows=600_000)
test_full = pd.read_csv("data/amazon_review_polarity_csv/test.csv", header=None, names=cols)
train = train_full.sample(60_000, random_state=0).reset_index(drop=True)
test = test_full.sample(20_000, random_state=0).reset_index(drop=True)
for df in (train, test):
    df["review"] = df["title"].fillna("") + ". " + df["text"].fillna("")    # the title is part of the review
    df["positive"] = (df["label"] == 2).astype(int)
print(train.shape, test.shape)
print("share positive:", train["positive"].mean().round(3), test["positive"].mean().round(3))
train[["positive", "review"]].head()

The classes are balanced (half positive), so accuracy is a fair summary here, unlike in the fraud lab; we still report precision, recall and F1 for each class.

## 2. Text preprocessing

Every document goes through the same steps:
1. **Lowercase**: "Great" and "great" become the same word.
2. **Remove punctuation and digits**: keep letters only.
3. **Remove stopwords**: very frequent words (the, and, is, ...) that carry little meaning on their own.
4. **Stem**: cut words to a common root with the Porter stemmer, so that "loved", "loving" and "loves" all become "love".

We make each step switchable, because we will measure what each one contributes.

In [ ]:
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stemmer = PorterStemmer()
stem_cache = {}

def stem(word):
    if word not in stem_cache:               # stemming the same word again and again is slow; remember results
        stem_cache[word] = stemmer.stem(word)
    return stem_cache[word]

def preprocess(text, remove_stopwords=True, use_stemming=True, stopwords=ENGLISH_STOP_WORDS):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)
    tokens = text.split()
    if remove_stopwords:
        tokens = [t for t in tokens if t not in stopwords]
    if use_stemming:
        tokens = [stem(t) for t in tokens]
    return " ".join(tokens)

example = train["review"][3]
print("original :", example[:200])
print("processed:", preprocess(example)[:200])

**A warning about stopword lists.** scikit-learn's English list contains words that matter enormously for sentiment:

In [ ]:
print([w for w in ["not", "no", "never", "nor", "nothing", "cannot", "very", "too", "well", "against"] if w in ENGLISH_STOP_WORDS])

"not good" and "good" become identical once "not" is removed. We therefore also try a **sentiment-safe** stopword list that keeps the negations and intensifiers.

In [ ]:
keep_words = {"not", "no", "never", "nor", "nothing", "cannot", "very", "too", "against", "none", "neither", "without"}
safe_stopwords = frozenset(ENGLISH_STOP_WORDS - keep_words)

t = time.time()
variants = {
    "raw (lowercase, letters only)": dict(remove_stopwords=False, use_stemming=False),
    "+ stopwords removed (standard list)": dict(remove_stopwords=True, use_stemming=False),
    "+ stopwords removed (keeping negations)": dict(remove_stopwords=True, use_stemming=False, stopwords=safe_stopwords),
    "+ stemming only": dict(remove_stopwords=False, use_stemming=True),
    "+ stopwords (keeping negations) + stemming": dict(remove_stopwords=True, use_stemming=True, stopwords=safe_stopwords),
}
processed = {name: (train["review"].map(lambda s: preprocess(s, **kw)), test["review"].map(lambda s: preprocess(s, **kw)))
             for name, kw in variants.items()}
print(f"preprocessed {len(variants)} variants in {time.time() - t:.0f}s")

## 3. TF-IDF representation

Each review becomes a vector with one entry per word (or word pair) of the vocabulary. **TF-IDF** weights each entry:

$$\text{tf-idf}(t, d) = \text{tf}(t, d) \times \text{idf}(t), \qquad \text{idf}(t) = \ln\frac{1 + n}{1 + \text{df}(t)} + 1$$

where $\text{tf}(t, d)$ is how often word $t$ appears in review $d$, $n$ is the number of reviews and $\text{df}(t)$ the number of reviews containing $t$. Each vector is then scaled to length 1.

**Why it works:**
- A word that appears in almost every review ("product", "one", "get") can't tell reviews apart; its IDF is small, so it counts little.
- A word that appears in few reviews but often in this one ("refund", "flimsy", "masterpiece") is a strong clue about this review; its IDF is large.
- Scaling to length 1 removes the effect of review length: a long rave and a short rave look alike.
- `sublinear_tf=True` uses $1 + \ln(\text{tf})$ instead of tf, so the tenth "great" adds less than the first.

We use single words and pairs of consecutive words (**bigrams**), so that "not good", "waste money" and "highly recommend" become features of their own; and ignore terms that appear in fewer than 3 reviews.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

x_train_text, x_test_text = processed["+ stopwords (keeping negations) + stemming"]
vectorizer = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)
X_train = vectorizer.fit_transform(x_train_text)          # vocabulary and IDF learned from training reviews only
X_test = vectorizer.transform(x_test_text)
y_train, y_test = train["positive"], test["positive"]
print("vocabulary size:", X_train.shape[1], "| non-zero entries per review on average:", round(X_train.nnz / X_train.shape[0]))
idf = pd.Series(vectorizer.idf_, index=vectorizer.get_feature_names_out())
print("lowest IDF (most common):", idf.nsmallest(6).round(2).to_dict())
print("examples of high IDF:", idf.sample(6, random_state=1).round(2).to_dict())

## 4. Models

- **Multinomial Naive Bayes**: applies Bayes' theorem, assuming the words are independent given the class. Very fast; works directly with counts or TF-IDF values.
- **Logistic regression**: a linear model on the TF-IDF features, trained by minimising cross-entropy with an L2 penalty.
- **Linear SVM**: also linear, with the hinge loss: it looks for the widest margin between positive and negative reviews.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, f1_score, confusion_matrix

models = {"naive bayes": MultinomialNB(alpha=0.3),
          "logistic regression": LogisticRegression(C=4, max_iter=2000),
          "linear SVM": LinearSVC(C=0.5)}
preds = {}
for name, model in models.items():
    t = time.time()
    model.fit(X_train, y_train)
    preds[name] = model.predict(X_test)
    print(f"{name:20s} accuracy {accuracy_score(y_test, preds[name]):.4f}  F1 {f1_score(y_test, preds[name]):.4f}  ({time.time() - t:.1f}s)")

## 5. Evaluation

In [ ]:
best = max(preds, key=lambda n: accuracy_score(y_test, preds[n]))
print("best:", best)
print(classification_report(y_test, preds[best], target_names=["negative", "positive"], digits=3))
print(pd.DataFrame(confusion_matrix(y_test, preds[best]), index=["actual neg", "actual pos"], columns=["pred neg", "pred pos"]))

### Failure cases
Which reviews does the best model get wrong, and how confidently? We use logistic regression's probability to find the reviews it is most confidently wrong about.

In [ ]:
lr = models["logistic regression"]
test_view = test[["positive", "review"]].copy()
test_view["p_positive"] = lr.predict_proba(X_test)[:, 1]
wrong = test_view[(test_view["p_positive"] >= 0.5) != (test_view["positive"] == 1)]
print(f"{len(wrong)} misclassified of {len(test_view)}")
print("\nNegative reviews the model was most sure were POSITIVE:")
for _, r in wrong[wrong["positive"] == 0].nlargest(5, "p_positive").iterrows():
    print(f"  p={r['p_positive']:.2f} | {r['review'][:220]}")
print("\nPositive reviews the model was most sure were NEGATIVE:")
for _, r in wrong[wrong["positive"] == 1].nsmallest(5, "p_positive").iterrows():
    print(f"  p={r['p_positive']:.2f} | {r['review'][:220]}")

Typical failure patterns (read the examples above and look for them):
- **Contrast and negation over long distances**: "I wanted to love this... the first chapters were great, but..." The positive words come first and the verdict comes later.
- **Sarcasm and irony**: positive words used to mean the opposite.
- **Comparisons**: praising another product in a negative review ("buy the old version instead, it was excellent").
- **Mislabelled data**: some reviews clearly contradict their star rating; no model can get those "right".

A bag of words (even with bigrams) can't follow the structure of a sentence; that is where sequence models and modern language models do better.

## 6. Deliverables

### Top positive and negative words (logistic regression weights)

In [ ]:
weights = pd.Series(lr.coef_[0], index=vectorizer.get_feature_names_out())
top = pd.DataFrame({"most POSITIVE": weights.nlargest(25).index, "weight +": weights.nlargest(25).values.round(2),
                    "most NEGATIVE": weights.nsmallest(25).index, "weight -": weights.nsmallest(25).values.round(2)})
top

The words are stemmed ("disappoint" covers disappointed, disappointing, disappointment). Note the bigrams: "not disappoint" and "not bad" are positive while "disappoint" alone is strongly negative. With the standard stopword list, which deletes "not", these would collapse into the plain words.

### Ablation: what does each preprocessing step contribute?
Same vectoriser and models, each preprocessing variant:

In [ ]:
rows = []
for variant, (tr_text, te_text) in processed.items():
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True)
    Xtr, Xte = vec.fit_transform(tr_text), vec.transform(te_text)
    row = {"preprocessing": variant, "vocabulary": Xtr.shape[1]}
    for name, model in [("NB", MultinomialNB(alpha=0.3)), ("logistic", LogisticRegression(C=4, max_iter=2000)), ("linear SVM", LinearSVC(C=0.5))]:
        row[f"{name} accuracy"] = accuracy_score(y_test, model.fit(Xtr, y_train).predict(Xte))
    rows.append(row)
ablation = pd.DataFrame(rows).set_index("preprocessing")
ablation.round(4)

### Write-up: the impact of preprocessing

The ablation table gives a clear answer on this data:
- **The least preprocessing is best.** Lowercase, letters only, no stopword removal, no stemming: 0.915 accuracy for the linear SVM, 0.914 for logistic regression.
- **Removing stopwords with the standard list hurts** every model, by about 2.5 points: the list deletes "not", "no", "never", "very", and with them bigrams such as "not good", "not worth" and "not disappoint" (which appear among the strongest features above).
- **Keeping the negations recovers only about half of that loss** (0.903). The other stopwords matter too, inside phrases: "waste of", "the best", "too much", "was ok".
- **Stemming makes almost no difference** (0.914 vs 0.915): with 60,000 reviews there is enough data to learn "disappointed" and "disappointing" separately, and stemming halves nothing that matters. It does shrink the vocabulary, which matters more with small data.
- **The choice of model matters less than the preprocessing**: logistic regression and the linear SVM are within 0.1–0.2 points of each other; Naive Bayes is about 3 points behind on every variant.

So the configuration we used in sections 3–5 (stopwords without negations, plus stemming) is *not* the best one: on this data the best model is the **linear SVM on TF-IDF of the raw lowercased text with unigrams and bigrams (91.5%)**.

Lesson: preprocessing steps are hypotheses to test, not rules. Stopword removal, standard advice for finding what a document is *about* (topic classification, search), hurts when the task is *how the writer feels about it*.

**Also note in the failure cases:** several "negative" reviews the model was sure were positive ("Great Bargain! ... I highly recommend it") are clearly positive reviews with a wrong label. Real datasets contain label noise, which sets a ceiling on any model's accuracy.